In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import *
from datetime import date, datetime

In [0]:
spark = SparkSession.builder.appName("Bronze_Data_Validation").getOrCreate()

In [0]:
%sql
CREATE TABLE workspace.default.rule_config 
(
  rule_id INT,
  table_name varchar(50),
  field_name varchar(50),
  rule_type VARCHAR(50),
  rule_expression VARCHAR(50),
  error_code VARCHAR(50),
  error_message VARCHAR(50),
  IS_active VARCHAR(1)
)

In [0]:
%sql
UPDATE workspace.default.rule_config  SET IS_active='N' WHERE rule_expression="REQUIRED"

In [0]:
%sql
SELECT * FROM workspace.default.rule_config 

In [0]:
%sql
INSERT INTO workspace.default.rule_config  VALUES(1,'student','student_id','STRUCTURE','IS NOT NULL','001','student_id is NULL','Y');
INSERT INTO workspace.default.rule_config  VALUES(2,'student','first_name','STRUCTURE','IS NOT NULL','001','First Name is NULL','Y');
INSERT INTO workspace.default.rule_config  VALUES(3,'student','dob','STRUCTURE','IS NOT NULL','001','dob is NULL','Y');
INSERT INTO workspace.default.rule_config  VALUES(4,'student','student_id','STRUCTURE','REQUIRED','002','student_id is required','Y');
INSERT INTO workspace.default.rule_config  VALUES(5,'student','TABLE_LOAD_DATE','STRUCTURE','REQUIRED','002','TABLE_LOAD_DATE is required','Y');

In [0]:
%sql
INSERT INTO workspace.default.rule_config  VALUES(6,'teacher','student_id','STRUCTURE','IS NOT NULL','001','student_id is NULL','Y');
INSERT INTO workspace.default.rule_config  VALUES(7,'teacher','first_name','STRUCTURE','IS NOT NULL','001','First Name is NULL','Y');

In [0]:
data = [
    ( "John", "Doe", date(2000, 1, 15)),
    ( "Jane", "Smith", date(2001, 5, 20)),
    ( "Peter", "Jones", date(1999, 8, 30))
]

# Column names
columns = ["first_name", "last_name", "dob"]


Define the Variables

In [0]:
Process_table_Name = "student"

Read from Bronze 

In [0]:
# Create the DataFrame
df_bronze = spark.createDataFrame(data, schema=columns)

# Show the DataFrame's content and its inferred schema
print("DataFrame Content:")
df_bronze.show()

print("Inferred DataFrame Schema:")
df_bronze.printSchema()

Read Rule Table 

In [0]:
df_Rule = spark.sql(f"""
                    SELECT * FROM workspace.default.rule_config where table_name = '{Process_table_Name}'
                    """)
df_Rule.show(truncate = False)                    

Rule -1 REQUIRED Column Check

In [0]:
Process_table_Name = "student"

In [0]:
import sys

In [0]:
def required_column_check(df_bronze, df_Rule, Process_table_Name):
    required_rules = df_Rule.filter(
                                    (col("rule_type") == "STRUCTURE") &
                                    (col("IS_active") == "Y") &
                                    (col("rule_expression")== "REQUIRED") &
                                    (col("table_name") == Process_table_Name)
    )

    has_required_rules = required_rules.limit(1).count() > 0
    if has_required_rules:
        required_cols = [row["field_name"] for row in required_rules.select("field_name").collect()]
        missing_cols = [ c for c in required_cols if c not in df_bronze.columns]
        if missing_cols:
            cols_str = ", ".join(missing_cols)
            error_msgs = f"Missing fields {cols_str} is requried"
            error_code = required_rules.select("error_code").first()["error_code"]
            failed_df = df_bronze.withColumn("error_code", lit(error_code))\
                             .withColumn("error_msgs",lit(error_msgs))

            success_df = df_bronze.limit(0)
            print("Requried Columns are missing in bronze data")
            return "FAIL",failed_df,success_df
        else:
            print("No requried columns are missing")
            return "PASS", None, None
    else :
        print("No required column rules found --> skipping required column check")
        return "SKIP", None, None
    

In [0]:
status,failed_df,success_df = required_column_check(df_bronze, df_Rule, Process_table_Name)
print(status)
display(failed_df)
display(success_df)

In [0]:
if status == "FAIL":
    print("Requried column validation Failed")
    raise Exception("Stopping job due to required column validation failure")
else:
    print("proceeding to null check validation")